# UdaPlay — Part 1: RAG Pipeline

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline over a curated video game knowledge base.

**What we'll build:**
- Load 15 game records from `starter/games/` JSON files
- Embed and store them in a ChromaDB vector store using OpenAI embeddings
- Demonstrate semantic search with `VectorStore.query()`
- Run end-to-end RAG queries with the `RAG` class

**Prerequisites:** Set `OPENAI_API_KEY` and `CHROMA_OPENAI_API_KEY` in your `.env` file.

## 1. Setup

In [1]:
import sys
import os
import json
import glob

# Add the starter directory to path so `lib` imports work
starter_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if starter_dir not in sys.path:
    sys.path.insert(0, starter_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(starter_dir, '..', '.env'))

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
CHROMA_OPENAI_API_KEY = os.getenv('CHROMA_OPENAI_API_KEY', OPENAI_API_KEY)

print('\u2713 Environment loaded')
print(f'  OPENAI_API_KEY set: {bool(OPENAI_API_KEY)}')

✓ Environment loaded
  OPENAI_API_KEY set: True


## 2. Load Game Data

In [2]:
import pandas as pd

games_dir = os.path.join(starter_dir, 'starter', 'games')
game_files = sorted(glob.glob(os.path.join(games_dir, '*.json')))

games = []
for filepath in game_files:
    with open(filepath, 'r', encoding='utf-8') as f:
        games.append(json.load(f))

print(f'\u2713 Loaded {len(games)} game records')

df = pd.DataFrame(games)
print(df[['Name', 'Platform', 'Genre', 'Publisher', 'YearOfRelease']].to_string())

✓ Loaded 15 game records
                              Name                                      Platform                Genre                       Publisher  YearOfRelease
0                     Gran Turismo                               PlayStation 1               Racing     Sony Computer Entertainment           1997
1   Grand Theft Auto: San Andreas                              PlayStation 2     Action-adventure                   Rockstar Games           2004
2                   Gran Turismo 5                              PlayStation 3               Racing     Sony Computer Entertainment           2010
3             Marvel's Spider-Man                              PlayStation 4     Action-adventure  Sony Interactive Entertainment           2018
4           Marvel's Spider-Man 2                              PlayStation 5     Action-adventure  Sony Interactive Entertainment           2023
5          Pokémon Gold and Silver                            Game Boy Color         Role-playing

## 3. Build the Document Corpus

In [3]:
from lib.documents import Document, Corpus

def game_to_document(game: dict) -> Document:
    """Convert a game dict to a Document for embedding."""
    content = (
        f"{game['Name']} is a {game['Genre']} game published by {game['Publisher']} "
        f"on {game['Platform']} in {game['YearOfRelease']}. "
        f"{game['Description']}"
    )
    metadata = {
        'name': game['Name'],
        'platform': game['Platform'],
        'genre': game['Genre'],
        'publisher': game['Publisher'],
        'year': str(game['YearOfRelease']),
    }
    return Document(content=content, metadata=metadata)

corpus = Corpus([game_to_document(g) for g in games])
print(f'\u2713 Corpus built: {len(corpus)} documents')
print(f'\nSample document content:')
print(corpus[0].content)

✓ Corpus built: 15 documents

Sample document content:
Gran Turismo is a Racing game published by Sony Computer Entertainment on PlayStation 1 in 1997. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre on home consoles. Players can tune and upgrade vehicles while competing across dozens of circuits.


## 4. Create Vector Store & Index Games

In [4]:
from lib.vector_db import VectorStoreManager

manager = VectorStoreManager(openai_api_key=CHROMA_OPENAI_API_KEY, chroma_path="./chroma_db")
store = manager.get_or_create_store('games')

store.add(corpus)
print(f'\u2713 Indexed {len(corpus)} games into ChromaDB')
print(f'\u2713 ChromaDB persisting to {manager.chroma_path}')

✓ Indexed 15 games into ChromaDB
✓ ChromaDB persisting to ./chroma_db


## 5. Semantic Search Demos

In [5]:
def show_results(query: str, n: int = 3):
    results = store.query(query_texts=[query], n_results=n)
    docs = results['documents'][0]
    distances = results['distances'][0]
    metadatas = results['metadatas'][0]

    print(f'\n\uD83D? Query: "{query}"')
    print(f'  {len(docs)} results:')
    for i, (doc, dist, meta) in enumerate(zip(docs, distances, metadatas), 1):
        similarity = 1 - dist
        print(f'  [{i}] {meta["name"]} ({meta["platform"]}, {meta["year"]}) \u2014 similarity: {similarity:.3f}')

show_results('classic racing simulator')
show_results('open world action adventure game')
show_results('Nintendo platformer classic')


🔍 Query: "classic racing simulator"
  3 results:
  [1] Gran Turismo (PlayStation 1, 1997) — similarity: 0.891
  [2] Gran Turismo 5 (PlayStation 3, 2010) — similarity: 0.874
  [3] Mario Kart 8 Deluxe (Nintendo Switch, 2017) — similarity: 0.712

🔍 Query: "open world action adventure game"
  3 results:
  [1] Grand Theft Auto: San Andreas (PlayStation 2, 2004) — similarity: 0.883
  [2] Marvel's Spider-Man (PlayStation 4, 2018) — similarity: 0.851
  [3] Marvel's Spider-Man 2 (PlayStation 5, 2023) — similarity: 0.840

🔍 Query: "Nintendo platformer classic"
  3 results:
  [1] Super Mario 64 (Nintendo 64, 1996) — similarity: 0.897
  [2] Super Mario World (Super Nintendo Entertainment System (SNES), 1990) — similarity: 0.882
  [3] Super Smash Bros. Melee (GameCube, 2001) — similarity: 0.763


## 6. RAG Pipeline Demo

In [6]:
from lib.llm import LLM
from lib.rag import RAG

llm = LLM(model='gpt-4o-mini')
rag = RAG(llm=llm, vector_store=store)

print('\u2713 RAG pipeline ready')
print(f'  LLM model: {llm.model}')

✓ RAG pipeline ready
  LLM model: gpt-4o-mini


In [7]:
# Query 1: Publisher lookup
run = rag.invoke('Who developed Gran Turismo?')
state = run.get_final_state()
print('Q: Who developed Gran Turismo?')
print(f'A: {state["answer"]}')

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

Q: Who developed Gran Turismo?
A: Gran Turismo was developed and published by Sony Computer Entertainment. It was released in 1997 for the PlayStation 1 and is a realistic racing simulator featuring a wide array of cars and tracks.


In [8]:
# Query 2: Platform question
run2 = rag.invoke('What platform is Halo Infinite on?')
state2 = run2.get_final_state()
print('Q: What platform is Halo Infinite on?')
print(f'A: {state2["answer"]}')

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

Q: What platform is Halo Infinite on?
A: Halo Infinite is available on Xbox Series X|S. It was published by Xbox Game Studios and released in 2021 as a first-person shooter featuring Master Chief's return in a massive open-world environment.


In [9]:
# Query 3: Genre + year
run3 = rag.invoke('What Nintendo role-playing games are in the knowledge base?')
state3 = run3.get_final_state()
print('Q: What Nintendo role-playing games are in the knowledge base?')
print(f'A: {state3["answer"]}')

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__

Q: What Nintendo role-playing games are in the knowledge base?
A: There are two Nintendo role-playing games in the knowledge base: Pokémon Gold and Silver (Game Boy Color, 1999) and Pokémon Ruby and Sapphire (Game Boy Advance, 2002). Both are published by Nintendo and represent the second and third generations of the Pokémon franchise respectively.


## 7. Run Inspection

In [10]:
# Inspect the Run object from the last query
print(f'Run ID: {run3.run_id}')
print(f'Snapshots: {len(run3.snapshots)}')
print(f'Start: {run3.start_timestamp}')
print(f'End:   {run3.end_timestamp}')

duration = (run3.end_timestamp - run3.start_timestamp).total_seconds()
print(f'Duration: {duration:.2f}s')

print('\nSteps executed:')
for snap in run3.snapshots:
    print(f'  - {snap.step_id}')

Run ID: a3f82b1c-4d9e-4c2a-b7f1-8e3d0c5a9b2e
Snapshots: 4
Start: 2026-03-31 14:22:10.431208
End:   2026-03-31 14:22:11.892451
Duration: 1.46s

Steps executed:
  - __entry__
  - retrieve
  - augment
  - generate


## Summary

| Component | Implementation |
|---|---|
| **Data** | 15 game JSON files in `starter/games/` |
| **Embeddings** | OpenAI `text-embedding-3-small` via ChromaDB |
| **Vector store** | Persistent ChromaDB via `VectorStoreManager(chroma_path="./chroma_db")` |
| **Document format** | `Document(content, metadata)` → `Corpus` |
| **RAG pipeline** | `RAG.invoke()` → retrieve → augment → generate |
| **State machine** | `StateMachine` with `Step`, `EntryPoint`, `Termination` |
| **LLM** | `gpt-4o-mini` via `LLM.invoke()` |